In [1]:
import sqlite3
from datetime import date

def init_database():
    conn = sqlite3.connect('university.db')
    cursor = conn.cursor()
    
    # Enable foreign key support (SQLite needs this for each connection)
    cursor.execute("PRAGMA foreign_keys = ON")
    
    # Create Emp table (must come first due to circular reference)
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS emp (
        eno TEXT(4) PRIMARY KEY,
        ename TEXT(50),
        birthday DATE,
        level INTEGER DEFAULT 3 CHECK(level BETWEEN 1 AND 5),
        position TEXT(10) CHECK(position IN ('教师', '教务', '会计', '秘书')),
        salary REAL CHECK(salary BETWEEN 2000 AND 200000),
        dno TEXT(4),
        FOREIGN KEY (dno) REFERENCES dept(dno) DEFERRABLE INITIALLY DEFERRED
    )
    ''')
    
    # Create Dept table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dept (
        dno TEXT(4) PRIMARY KEY,
        dname TEXT(20) CHECK(dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')),
        budget REAL,
        manager TEXT(4),
        FOREIGN KEY (manager) REFERENCES emp(eno) DEFERRABLE INITIALLY DEFERRED
    )
    ''')
    
    conn.commit()
    return conn

In [3]:
def insert_with_circular_reference(conn):
    cursor = conn.cursor()
    
    try:
        # Start a transaction
        cursor.execute("BEGIN TRANSACTION")
        
        # Insert department first (manager will be set later)
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)",
            ('D001', '计算机学院', 5000000, None)
        )
        
        # Insert employee referencing the department
        cursor.execute(
            "INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)",
            ('E001', '张教授', date(1975, 5, 15), 4, '教师', 80000, 'D001')
        )
        
        # Now update the department to reference the employee as manager
        cursor.execute(
            "UPDATE dept SET manager = ? WHERE dno = ?",
            ('E001', 'D001')
        )
        
        conn.commit()
        print("Successfully inserted circular references")
        
    except sqlite3.Error as e:
        conn.rollback()
        print("Failed to insert circular references:", e)

def test_constraints(conn):
    cursor = conn.cursor()
    
    print("\nTesting valid inserts:")
    try:
        # Valid employee
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E002', '李老师', 3, '教师', 50000)
        )
        print("- Valid employee inserted")
        
        # Valid department
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget) VALUES (?, ?, ?)",
            ('D002', '数学学院', 3000000)
        )
        print("- Valid department inserted")
        
    except sqlite3.Error as e:
        print("Valid insert failed:", e)
    
    print("\nTesting invalid inserts:")
    
    # Invalid level
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E003', 'Invalid', 6, '教师', 50000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid level (6):", e)
    
    # Invalid position
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E004', 'Invalid', 3, '校长', 50000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid position (校长):", e)
    
    # Invalid salary
    try:
        cursor.execute(
            "INSERT INTO emp (eno, ename, level, position, salary) VALUES (?, ?, ?, ?, ?)",
            ('E005', 'Invalid', 3, '教师', 1000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid salary (1000):", e)
    
    # Invalid department name
    try:
        cursor.execute(
            "INSERT INTO dept (dno, dname, budget) VALUES (?, ?, ?)",
            ('D003', '物理学院', 2000000)
        )
    except sqlite3.IntegrityError as e:
        print("- Caught invalid department name (物理学院):", e)
    
    conn.rollback()  # Rollback all test data

def test():
    conn = init_database()
    
    # Test circular reference insertion
    print("\n=== Testing Circular Reference ===")
    insert_with_circular_reference(conn)
    
    # Test constraints
    print("\n=== Testing Constraints ===")
    test_constraints(conn)
    
    # Verify the circular reference was successful
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM emp WHERE eno = 'E001'")
    emp = cursor.fetchone()
    print("\nEmployee E001:", emp)
    
    cursor.execute("SELECT * FROM dept WHERE dno = 'D001'")
    dept = cursor.fetchone()
    print("Department D001:", dept)
    
    conn.close()


In [8]:
test()


=== 测试互相引用约束 ===
成功插入互相引用的行

=== 测试工资级别约束 ===
正测试: 成功插入有效数据 (level=2, salary=8000)
负测试: 成功捕获违反约束的操作 - CHECK constraint failed: (level = 1 AND salary BETWEEN 0 AND 5000) OR
            (level = 2 AND salary BETWEEN 5001 AND 10000) OR
            (level = 3 AND salary BETWEEN 10001 AND 15000) OR
            (level = 4 AND salary BETWEEN 15001 AND 20000) OR
            (level = 5 AND salary > 20000)

=== 测试智能码生成 ===
生成的员工智能码: 020102198504040018

智能码解析:
员工编号: 0201
部门编码: 02 (市场部)
出生年份: 1985
职位编码: 04 (总监)
级别编码: 04 (4)
工资等级: 0018 (18)
